# 03 — Equilibrium Analysis & Comparative Statics

This notebook studies how the MFG equilibrium changes with model parameters
(*comparative statics*), verifies the Nash equilibrium property (ε-Nash), and
analyses stability via the Picard map's spectral radius.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from mfglob.grids import Grid1D, TimeGrid
from mfglob.models.lob_formation import LOBFormationMFG
from mfglob.mfg_solver import MFGSolver
from mfglob.equilibrium import (
    EquilibriumAnalyzer, compute_social_cost, compute_price_of_anarchy)


## 1. Solve base equilibrium

In [ ]:
model  = LOBFormationMFG(c_far=0.1, c_near=1.0, kappa=2.0,
                        c_crowd=0.3, c_control=0.1, sigma=0.5)
grid   = Grid1D(0.0, 8.0, 60)
tgrid  = TimeGrid(0.5, 50)
sol    = MFGSolver(model, grid, tgrid, damping=0.5, tol=1e-3,
                   max_iterations=50).solve(verbose=False)
print(f"Converged: {sol['converged']}  ({sol['n_iterations']} iters)")

ea     = EquilibriumAnalyzer(sol, model, grid, tgrid)
result = ea.verify_equilibrium()
inner  = result['checks']

print(f"Overall passed: {result['passed']}")
print(f"Details: {result['details']}")
print("\nIndividual checks:")
print(f"  {'metric':<35s}  value")
print("  " + "-" * 52)
for k, v in inner.items():
    if isinstance(v, bool):
        print(f"  {k:<35s}  {'PASS' if v else 'FAIL'}")
    else:
        print(f"  {k:<35s}  {v:.4e}")


## 2. ε-Nash verification

In [ ]:
eps = ea.epsilon_nash_check(n_perturbations=50, perturbation_scale=0.05)
print("ε-Nash check:")
print(f"  epsilon:                 {eps['epsilon']:.6f}")
print(f"  all_perturbations_worse: {eps['all_perturbations_worse']}")
print(f"  baseline_cost:           {eps['baseline_cost']:.6f}")


## 3. Social cost and Price of Anarchy

PoA = Nash cost / cooperative cost.

In [ ]:
sc_nash = compute_social_cost(sol, model, grid, tgrid)
poa     = compute_price_of_anarchy(model, grid, tgrid)

print(f"Nash social cost:        {sc_nash:.6f}")
print(f"Cooperative social cost: {poa['social_cost_optimum']:.6f}")
print(f"Price of Anarchy:        {poa['price_of_anarchy']:.4f}")


## 4. Comparative statics — c_crowd

In [ ]:
c_crowds = [0.05, 0.1, 0.3, 0.5, 1.0, 2.0]
rows = []
for c_c in c_crowds:
    m_loc  = LOBFormationMFG(c_crowd=c_c)
    g_loc  = Grid1D(0.0, 8.0, 50)
    tg_loc = TimeGrid(0.5, 40)
    s_loc  = MFGSolver(m_loc, g_loc, tg_loc, damping=0.5,
                       tol=1e-3, max_iterations=40).solve(verbose=False)
    x_l   = g_loc.points
    m_avg = np.mean(s_loc['density'], axis=0)
    m_avg /= np.trapezoid(m_avg, x_l)
    mode  = float(x_l[np.argmax(m_avg)])
    mean  = float(np.trapezoid(x_l * m_avg, x_l))
    sc    = compute_social_cost(s_loc, m_loc, g_loc, tg_loc)
    rows.append({'c_crowd': c_c, 'mode': mode, 'mean': mean, 'sc': sc})

print(f"{'c_crowd':>10}  {'mode':>8}  {'mean':>8}  {'social_cost':>13}")
print("-" * 44)
for r in rows:
    print(f"{r['c_crowd']:>10.3f}  {r['mode']:>8.4f}  "
          f"{r['mean']:>8.4f}  {r['sc']:>13.6f}")


## 5. Stability analysis

In [ ]:
stab = ea.stability_analysis()
print('Stability of Picard map at equilibrium:')
print(f"  Spectral radius: {stab['spectral_radius']:.6f}")
print(f"  Stable:          {stab['stable']}")
print(f"  Spectral gap:    {stab['spectral_gap']:.6f}")
print('Spectral radius < 1 confirms local convergence.')
